In [2]:
!pip install -q "zarr>=3.0.10" "tracksdata @ git+https://github.com/royerlab/tracksdata@main" scipy tqdm

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import torch
import zarr
import scipy
import numpy as np
import tracksdata

print("PyTorch    :", torch.__version__)
print("Zarr       :", zarr.__version__)
print("SciPy      :", scipy.__version__)
print("NumPy      :", np.__version__)
print("tracksdata :", "OK")

In [ ]:
import json
import torch
from pathlib import Path

config_path = Path(
    "/kaggle/input/notebooks/thibautgoldsborough/"
    "unet-baseline-inference-submission/"
    "repo/weights/unet_transformer/split_0/config.json"
)

weights_path = Path(
    "/kaggle/input/notebooks/thibautgoldsborough/"
    "unet-baseline-inference-submission/"
    "repo/weights/unet_transformer/split_0/edge_predictor_best.pth"
)

with open(config_path) as f:
    config = json.load(f)

checkpoint = torch.load(
    weights_path,
    map_location="cpu",
    weights_only=False
)

print("Config:")
for k, v in config.items():
    print(f"  {k}: {v}")

print("\nCheckpoint tensors:", len(checkpoint))
print("First 5:")
for k in list(checkpoint.keys())[:5]:
    print(f"  {k}: {tuple(checkpoint[k].shape)}")

In [ ]:
## Load the Released Baseline Model

The released checkpoint contains the weights for the complete tracking model:

1. Temporal U-Net — extracts spatiotemporal features.
2. Detection head — predicts cell centers.
3. Node transformer — compares detected cells between consecutive frames.
4. Pairwise MLP — predicts whether two cells should be connected.

We reconstruct the same architecture and load the released weights with strict key matching.

In [1]:
from tracking_cellmot.models.temporal_unet import TemporalUNet3D
from tracking_cellmot.models.simple_node_transformer import SimpleNodeTransformer
from tracking_cellmot.models.temporal_unet import TemporalUNet3D
from tracking_cellmot.models.simple_node_transformer import SimpleNodeTransformer
import torch.nn as nn


class UNetNodeTransformer(nn.Module):
    def __init__(
        self,
        unet_out_channels=32,
        unet_layers=(32, 64, 128),
    ):
        super().__init__()

        self.unet = TemporalUNet3D(
            in_channels=1,
            out_channels=unet_out_channels,
            layers=unet_layers,
        )

        self.detect_head = nn.Conv3d(
            unet_out_channels,
            1,
            kernel_size=1,
        )

        self.transformer = SimpleNodeTransformer(
            feat_dim=unet_out_channels + 32,
        )


model = UNetNodeTransformer(
    unet_out_channels=config["unet_out_channels"],
    unet_layers=tuple(config["unet_layers"]),
)

print("Complete model created.")
print("Parameters:", sum(p.numel() for p in model.parameters()))

ModuleNotFoundError: No module named 'tracking_cellmot'

## Scan multiple samples for a denser one

The first sample alphabetically has only 1 cell/frame (a single tracked lineage), so
spacing can't be computed on it. Sparse single-lineage samples are common in this
dataset. Scan several samples and pick the densest one to measure real cell size from.

In [ ]:
def find_data_dirs(root="/kaggle/input"):
    root = Path(root)
    if not root.exists() or not any(root.iterdir()):
        raise FileNotFoundError(
            "Nothing found under /kaggle/input. The competition dataset is not attached to "
            "this notebook. Click '+ Add Input' in the right sidebar, search for "
            "'Biohub - Cell Tracking During Development', and add it, then re-run this cell."
        )

    print("Contents of /kaggle/input:")
    for p in sorted(root.iterdir()):
        print(" ", p)

    train_candidates = [p for p in root.rglob("train") if p.is_dir() and any(p.glob("*.zarr"))]
    test_candidates = [p for p in root.rglob("test") if p.is_dir() and any(p.glob("*.zarr"))]

    if not train_candidates or not test_candidates:
        print("\nCould not auto-find train/test dirs containing .zarr folders.")
        print("All .zarr paths found anywhere under /kaggle/input:")
        for p in sorted(root.rglob("*.zarr"))[:20]:
            print(" ", p)
        raise FileNotFoundError(
            "No valid train/test directories with .zarr data found. Inspect the printed "
            "listing above and adjust find_data_dirs() if the folder layout differs."
        )

    train_dir = train_candidates[0]
    test_dir = test_candidates[0]
    print(f"\nUsing train dir: {train_dir}")
    print(f"Using test dir:  {test_dir}")
    return train_dir, test_dir

train_dir, test_dir = find_data_dirs()

In [ ]:

import json
from pathlib import Path

config_path = Path(
    "/kaggle/input/notebooks/thibautgoldsborough/"
    "unet-baseline-inference-submission/"
    "repo/weights/unet_transformer/split_0/config.json"
)

with open(config_path) as f:
    config = json.load(f)

print(json.dumps(config, indent=2))


In [ ]:
import torch
import zarr
import scipy
import numpy as np

print("PyTorch :", torch.__version__)
print("Zarr    :", zarr.__version__)
print("SciPy   :", scipy.__version__)
print("NumPy   :", np.__version__)

try:
    import tracksdata
    print("tracksdata: OK")
except Exception as e:
    print("tracksdata: ERROR")
    print(e)

In [ ]:
from scipy.spatial import cKDTree

def inspect_cell_sizes(geff_path):
    graph, _ = geff.read(str(geff_path), backend="networkx")
    by_t = {}
    for n, attrs in graph.nodes(data=True):
        by_t.setdefault(attrs["t"], []).append((attrs["z"], attrs["y"], attrs["x"]))

    dists = []
    for t, pts in by_t.items():
        if len(pts) < 2:
            continue
        tree = cKDTree(pts)
        d, _ = tree.query(pts, k=2)
        dists.extend(d[:, 1])

    dists = np.array(dists)
    if len(dists) == 0:
        print("Not enough co-occurring points per frame to estimate spacing.")
        return dists

    print(f"Nearest-neighbor cell spacing (voxels): "
          f"min={dists.min():.1f} p25={np.percentile(dists,25):.1f} "
          f"median={np.median(dists):.1f} p75={np.percentile(dists,75):.1f}")
    return dists


_sample_zarr = sorted(train_dir.glob("*.zarr"))[0]
_sample_geff = _sample_zarr.with_suffix(".geff")
if _sample_geff.exists():
    _ = inspect_cell_sizes(_sample_geff)
else:
    print(f"No .geff found next to {_sample_zarr}, skipping size inspection.")


In [ ]:
def diagnose_geff(geff_path):
    graph, _ = geff.read(str(geff_path), backend="networkx")
    by_t = {}
    for n, attrs in graph.nodes(data=True):
        by_t.setdefault(attrs["t"], []).append(n)

    counts = {t: len(nodes) for t, nodes in by_t.items()}
    print(f"{geff_path.name}: {graph.number_of_nodes()} total nodes, "
          f"{len(counts)} timepoints with >=1 node")
    print(f"  nodes/frame: min={min(counts.values())} "
          f"max={max(counts.values())} mean={np.mean(list(counts.values())):.1f}")
    return counts


candidates = sorted(train_dir.glob("*.zarr"))[:10]
best_sample, best_max_count = None, 0

for zp in candidates:
    gp = zp.with_suffix(".geff")
    if not gp.exists():
        continue
    counts = diagnose_geff(gp)
    max_count = max(counts.values())
    if max_count > best_max_count:
        best_max_count = max_count
        best_sample = gp

print(f"\nBest sample for spacing stats: {best_sample} (max {best_max_count} cells/frame)")
if best_sample:
    _ = inspect_cell_sizes(best_sample)

## Measure actual cell size from pixel data

Spacing between cells isn't cell size. Crop a small window around each known cell
center, threshold locally, and measure the connected-component size touching the
center — this gives a real, image-derived voxel count for `min_area`/`max_area`.

In [ ]:
from scipy.ndimage import label as cc_label

def measure_cell_footprints(zarr_path, geff_path, window=8, max_points=30):
    """Crop a small window around each labeled GT centroid, threshold locally,
    and measure the connected-component size touching the center voxel."""
    z = zarr.open(str(zarr_path), mode="r")
    vol = np.asarray(z["0"])

    graph, _ = geff.read(str(geff_path), backend="networkx")

    areas = []
    points = list(graph.nodes(data=True))[:max_points]

    for n, attrs in points:
        t, z_, y, x = attrs["t"], attrs["z"], attrs["y"], attrs["x"]
        frame = vol[t].astype(np.float32)

        z0, z1 = max(0, z_ - window), min(frame.shape[0], z_ + window)
        y0, y1 = max(0, y - window), min(frame.shape[1], y + window)
        x0, x1 = max(0, x - window), min(frame.shape[2], x + window)
        crop = frame[z0:z1, y0:y1, x0:x1]
        if crop.size == 0:
            continue

        smoothed = gaussian_filter(crop, sigma=1.0)
        try:
            thresh = threshold_otsu(smoothed)
        except ValueError:
            continue
        mask = smoothed > thresh

        labels, _ = cc_label(mask)
        center = (z_ - z0, y - y0, x - x0)
        center = tuple(int(np.clip(center[i], 0, mask.shape[i] - 1)) for i in range(3))
        center_label = labels[center]
        if center_label == 0:
            continue

        area = (labels == center_label).sum()
        areas.append(area)

    areas = np.array(areas)
    if len(areas) == 0:
        print("No valid footprints measured — try a different window size or sample.")
        return areas

    print(f"Measured {len(areas)} cell footprints (voxels): "
          f"min={areas.min()} p25={np.percentile(areas,25):.0f} "
          f"median={np.median(areas):.0f} p75={np.percentile(areas,75):.0f} "
          f"max={areas.max()}")
    return areas


areas = measure_cell_footprints(best_sample.with_suffix(".zarr"), best_sample)

## Measure real frame-to-frame movement

`max_distance` bounds how far ultrack will link a cell between consecutive frames.
Measure this directly from ground-truth tracks: for every real edge, how far did the
cell actually move?

In [ ]:
def measure_frame_to_frame_distance(geff_path, voxel_scale=VOXEL_SCALE):
    graph, _ = geff.read(str(geff_path), backend="networkx")
    coords = {n: np.array([attrs["z"], attrs["y"], attrs["x"]]) * np.array(voxel_scale)
              for n, attrs in graph.nodes(data=True)}

    dists = []
    for u, v in graph.edges():
        d = np.linalg.norm(coords[u] - coords[v])
        dists.append(d)

    dists = np.array(dists)
    if len(dists) == 0:
        print("No edges found in this sample.")
        return dists

    print(f"Frame-to-frame movement (micrometers): "
          f"min={dists.min():.2f} p25={np.percentile(dists,25):.2f} "
          f"median={np.median(dists):.2f} p75={np.percentile(dists,75):.2f} "
          f"max={dists.max():.2f}")
    return dists


move_dists = measure_frame_to_frame_distance(best_sample)

In [ ]:
def compute_foreground_contours(vol, smooth_sigma=1.0):
    """vol: (T, Z, Y, X) array. Returns (foreground, contours), both (T,Z,Y,X)."""
    T = vol.shape[0]
    foreground = np.zeros(vol.shape, dtype=bool)
    contours = np.zeros(vol.shape, dtype=np.float32)

    for t in range(T):
        frame = vol[t].astype(np.float32)
        smoothed = gaussian_filter(frame, sigma=smooth_sigma)

        try:
            thresh = threshold_otsu(smoothed)
        except ValueError:
            thresh = smoothed.max() * 0.5
        foreground[t] = smoothed > thresh

        edge = np.zeros_like(smoothed)
        for zi in range(smoothed.shape[0]):
            edge[zi] = sobel(smoothed[zi])
        e_min, e_max = edge.min(), edge.max()
        contours[t] = (edge - e_min) / (e_max - e_min + 1e-8)

    return foreground, contours
def run_ultrack(foreground, contours, voxel_scale=VOXEL_SCALE,
                 min_area=10, max_area=5000, max_distance=15.0):
    config = MainConfig()
    config.segmentation_config.min_area = min_area
    config.segmentation_config.max_area = max_area
    config.linking_config.max_distance = max_distance

    tracker = Tracker(config)
    tracker.track(foreground=foreground, edges=contours, scale=voxel_scale,
                   overwrite=True)

    tracks_df, graph = tracker.to_tracks_layer()
    return tracks_df, graph, tracker
def tracks_to_submission_rows(tracks_df, graph, dataset_name, row_id_start=0):
    rows = []
    row_id = row_id_start

    tracks_df = tracks_df.sort_values(["track_id", "t"]).reset_index(drop=True)
    node_id_map = {}
    next_node_id = 1

    for _, r in tracks_df.iterrows():
        key = (int(r["track_id"]), int(r["t"]))
        node_id_map[key] = next_node_id
        rows.append(dict(
            id=row_id, dataset=dataset_name, row_type="node",
            node_id=next_node_id, t=int(r["t"]),
            z=int(round(r["z"])), y=int(round(r["y"])), x=int(round(r["x"])),
            source_id=-1, target_id=-1,
        ))
        row_id += 1
        next_node_id += 1

    for track_id, grp in tracks_df.groupby("track_id"):
        grp = grp.sort_values("t")
        ts = grp["t"].tolist()
        for a, b in zip(ts[:-1], ts[1:]):
            src = node_id_map[(int(track_id), int(a))]
            tgt = node_id_map[(int(track_id), int(b))]
            rows.append(dict(
                id=row_id, dataset=dataset_name, row_type="edge",
                node_id=-1, t=-1, z=-1, y=-1, x=-1,
                source_id=src, target_id=tgt,
            ))
            row_id += 1

    tmin = tracks_df.groupby("track_id")["t"].min().to_dict()
    tmax = tracks_df.groupby("track_id")["t"].max().to_dict()
    for child_id, parent_id in graph.items():
        if parent_id is None or parent_id < 0:
            continue
        if parent_id not in tmax or child_id not in tmin:
            continue
        src = node_id_map.get((int(parent_id), int(tmax[parent_id])))
        tgt = node_id_map.get((int(child_id), int(tmin[child_id])))
        if src is None or tgt is None:
            continue
        rows.append(dict(
            id=row_id, dataset=dataset_name, row_type="edge",
            node_id=-1, t=-1, z=-1, y=-1, x=-1,
            source_id=src, target_id=tgt,
        ))
        row_id += 1

    return rows, row_id


def load_sample_volume(zarr_path):
    z = zarr.open(str(zarr_path), mode="r")
    return np.asarray(z["0"])

In [ ]:
def evaluate_one_sample(zarr_path, geff_gt_path, pred_geff_out,
                         min_area=10, max_area=5000, max_distance=15.0,
                         match_threshold_um=3.0):
    vol = load_sample_volume(zarr_path)
    foreground, contours = compute_foreground_contours(vol)
    tracks_df, graph, tracker = run_ultrack(
        foreground, contours, min_area=min_area, max_area=max_area,
        max_distance=max_distance,
    )

    shutil.rmtree(pred_geff_out, ignore_errors=True)
    tracker.to_geff(pred_geff_out)

    gt = load_geff_data(str(geff_gt_path), name="GT")
    pred = load_geff_data(str(pred_geff_out), name="pred")

    results, matched = run_metrics(
        gt_data=gt, pred_data=pred,
        matcher=PointMatcher(threshold=match_threshold_um),
        metrics=[BasicMetrics(), DivisionMetrics()],
    )
    return results


def evaluate_train_samples(train_root, n_samples=3, **kwargs):
    train_root = Path(train_root)
    zarr_paths = sorted(train_root.glob("*.zarr"))[:n_samples]

    all_results = []
    for zp in zarr_paths:
        sample_id = zp.stem
        geff_path = zp.with_suffix(".geff")
        if not geff_path.exists():
            print(f"skip {sample_id}: no .geff ground truth found")
            continue

        print(f"\n=== {sample_id} ===")
        pred_out = f"/kaggle/working/pred_{sample_id}.geff"
        results = evaluate_one_sample(zp, geff_path, pred_out, **kwargs)
        for r in results:
            print(f"  {r['metric']['name']}: {r['results']}")
        all_results.append((sample_id, results))

    return all_results